# 1 Likelihood Based Analysis of the 21-cm Power Spectrum


In [9]:
import numpy as np  
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [15]:
# Import simulation data
data_dir = Path("simulations")
files = sorted(data_dir.glob("*.npz"))

# Define test/train/validation split
num_files = len(files)
train_files = files[:int(0.8 * num_files)]
val_files = files[int(0.8 * num_files):int(0.9 * num_files)]
test_files = files[int(0.9 * num_files):]

# Write 
train_simulations = []
test_simulations = []
val_simulations = []

for f in train_files:
    with np.load(f, allow_pickle=True) as d:
        train_simulations.append(dict(d))

for f in val_files:
    with np.load(f, allow_pickle=True) as d:
        val_simulations.append(dict(d))

for f in test_files:
    with np.load(f, allow_pickle=True) as d:
        test_simulations.append(dict(d))

# Unpack
def unpack_simulations(simulations):
    params_list = []
    power_list  = []
    for sim in simulations:
        astro = sim['astro_params'].item()
        cosmo = sim['cosmo_params'].item()
        params_list.append([astro['L40_xray'], astro['fesc10'],
                            astro['epsstar'],  cosmo['h_fid']])
        power_list.append(sim['power'])
    return np.array(params_list), np.array(power_list)

raw_params_train, raw_power_train = unpack_simulations(train_simulations)
raw_params_val,   raw_power_val   = unpack_simulations(val_simulations)
raw_params_test,  raw_power_test  = unpack_simulations(test_simulations)

log_power_train = np.log(raw_power_train)
log_power_val   = np.log(raw_power_val)
log_power_test  = np.log(raw_power_test)



- We must normalise the training parameters as they can have different scales with respect to one another. We don't want to bias the trained NN based on the various input scales so we can scale them using a StandardScaler, which brings the mean to 0 and variance to 1.
- Each power spectrum mode has a different scale. For PCA, we want to find meaningful variations, not just the largest magnitude variations, which can be a result of different component scales. A component can have little relative variance in its axis but a huge absolute variance because of scale. As such, we normalise our power spectra as well.
- Since the power spectra vary across several orders of magnitude, it is prudent to work with the logarithm.

In [ ]:
power_train = np.log(raw_power_train)
power_val   = np.log(raw_power_val)
power_test  = np.log(raw_power_test)

# Normalise (Scale) datasets
params_scaler = StandardScaler().fit(raw_params_train)
power_scaler = StandardScaler().fit(log_power_train)

# Apply the same transformation to all datasets
params_train = params_scaler.transform(raw_params_train)
params_val   = params_scaler.transform(raw_params_val)
params_test  = params_scaler.transform(raw_params_test)
power_train = power_scaler.transform(log_power_train)
power_val   = power_scaler.transform(log_power_val)
power_test  = power_scaler.transform(log_power_test)

In [ ]:
dataset1 = np.load(files[0], allow_pickle=True)
dataset1.files
k = dataset1["k"]
power = dataset1["power"]
astro_params = dataset1["astro_params"]
cosmo_params = dataset1["cosmo_params"]
redshifts = dataset1["redshfit"]
code = dataset1["code"]
code_version = dataset1["code_version"]

for name, value in {
    "k": k,
    "power": power,
    "astro_params": astro_params,
    "cosmo_params": cosmo_params,
    "redshifts": redshifts,
    "code": code,
    "code_version": code_version,
}.items():
    print(f"{name}: {np.shape(value)}")

# print(k)
print(power)
print(np.log(min(power)), np.log(max(power)))
# print(astro_params)
# print(cosmo_params)
# print(redshifts)
# print(code)
# print(code_version)

k: (54,)
power: (54,)
astro_params: ()
cosmo_params: ()
redshifts: ()
code: ()
code_version: ()
[7.47685668e-03 1.17543845e-02 1.84238133e-02 2.87766302e-02
 4.47646190e-02 6.93055195e-02 1.06708847e-01 1.63245637e-01
 2.47878651e-01 3.73112822e-01 5.55905140e-01 8.18373566e-01
 1.18797717e+00 1.69641376e+00 2.37663902e+00 3.25690954e+00
 4.35249707e+00 5.65582583e+00 7.13087339e+00 8.71596303e+00
 1.03440688e+01 1.19720621e+01 1.36169846e+01 1.53657351e+01
 1.73573602e+01 1.97819180e+01 2.28448366e+01 2.66964759e+01
 3.11004660e+01 3.50089825e+01 3.71703915e+01 3.82915299e+01
 4.12093427e+01 4.74106189e+01 5.26737507e+01 5.31096109e+01
 5.61649336e+01 6.30732838e+01 6.40220110e+01 6.90476713e+01
 7.24870554e+01 7.67418995e+01 7.99308894e+01 8.42847893e+01
 8.80104033e+01 9.18137867e+01 9.56606479e+01 9.93856162e+01
 1.02964900e+02 1.06345587e+02 1.09440159e+02 1.12026565e+02
 1.13814238e+02 1.14498132e+02]
-4.895942805268216 4.740558504202362
